In [2]:
import pandas as pd

file_path = "../data/Product-Sales-Region.xlsx"

df = pd.read_excel(file_path)

df.head()

,Date,Region,Product,Quantity,UnitPrice,StoreLocation,CustomerType,Discount,Salesperson,TotalPrice,PaymentMethod,Promotion,Returned,OrderID,CustomerName,ShippingCost,OrderDate,DeliveryDate,RegionManager
0,2023-02-23,East,Laptop,14,163.60,Store B,Wholesale,0.00,Eva,2290.400,Online,FREESHIP,0,REG100000,Cust 6583,43.34,2023-02-23,2023-02-27,Eric
1,2024-12-19,South,Phone,1,544.01,Store A,Retail,0.00,Alice,544.010,Gift Card,SAVE10,0,REG100001,Cust 2144,5.30,2024-12-19,2024-12-28,Sophie
2,2023-05-10,North,Desk,14,346.18,Store B,Wholesale,0.10,Alice,4361.868,Online,WINTER15,0,REG100002,Cust 5998,20.46,2023-05-10,2023-05-19,Ryan
3,2025-02-26,Central,Chair,18,384.82,Store A,Wholesale,0.15,Frank,5887.746,Gift Card,FREESHIP,0,REG100003,Cust 7136,27.95,2025-02-26,2025-03-02,Cameron
4,2023-06-24,East,Desk,18,237.76,Store C,Retail,0.00,Carlos,4279.680,Online,SAVE10,0,REG100004,Cust 6506,5.73,2023-06-24,2023-06-27,Eric


In [3]:
import sys
!{sys.executable} -m pip install openpyxl


[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
df.shape   #dataset size


(1500, 19)

In [5]:
df.info()   #understand columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           1500 non-null   datetime64[ns]
 1   Region         1500 non-null   object        
 2   Product        1500 non-null   object        
 3   Quantity       1500 non-null   int64         
 4   UnitPrice      1500 non-null   float64       
 5   StoreLocation  1500 non-null   object        
 6   CustomerType   1500 non-null   object        
 7   Discount       1500 non-null   float64       
 8   Salesperson    1500 non-null   object        
 9   TotalPrice     1500 non-null   float64       
 10  PaymentMethod  1500 non-null   object        
 11  Promotion      1130 non-null   object        
 12  Returned       1500 non-null   int64         
 13  OrderID        1500 non-null   object        
 14  CustomerName   1500 non-null   object        
 15  ShippingCost   1500 n

In [6]:
df.columns

Index(['Date', 'Region', 'Product', 'Quantity', 'UnitPrice', 'StoreLocation',
       'CustomerType', 'Discount', 'Salesperson', 'TotalPrice',
       'PaymentMethod', 'Promotion', 'Returned', 'OrderID', 'CustomerName',
       'ShippingCost', 'OrderDate', 'DeliveryDate', 'RegionManager'],
      dtype='object')

In [7]:
df.isnull().sum()  #check missing values

Date               0
Region             0
Product            0
Quantity           0
UnitPrice          0
StoreLocation      0
CustomerType       0
Discount           0
Salesperson        0
TotalPrice         0
PaymentMethod      0
Promotion        370
Returned           0
OrderID            0
CustomerName       0
ShippingCost       0
OrderDate          0
DeliveryDate       0
RegionManager      0
dtype: int64

In [8]:
df["Promotion"] = df["Promotion"].fillna("No Promotion")

In [9]:
df.isnull().sum()

Date             0
Region           0
Product          0
Quantity         0
UnitPrice        0
StoreLocation    0
CustomerType     0
Discount         0
Salesperson      0
TotalPrice       0
PaymentMethod    0
Promotion        0
Returned         0
OrderID          0
CustomerName     0
ShippingCost     0
OrderDate        0
DeliveryDate     0
RegionManager    0
dtype: int64

In [10]:
df.duplicated().sum()  #check duplicates

np.int64(0)

In [11]:
df["OrderID"].duplicated().sum()

np.int64(0)

In [12]:
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["OrderID"], keep="first")
df["OrderID"].duplicated().sum()

np.int64(0)

In [13]:
date_columns = ["Date", "OrderDate", "DeliveryDate"]

for col in date_columns:
      df[col]   = pd.to_datetime(df[col])

print(df[date_columns].head())
print(df[date_columns].dtypes)

        Date  OrderDate DeliveryDate
0 2023-02-23 2023-02-23   2023-02-27
1 2024-12-19 2024-12-19   2024-12-28
2 2023-05-10 2023-05-10   2023-05-19
3 2025-02-26 2025-02-26   2025-03-02
4 2023-06-24 2023-06-24   2023-06-27
Date            datetime64[ns]
OrderDate       datetime64[ns]
DeliveryDate    datetime64[ns]
dtype: object


In [14]:
#verify no incorrect dates
df[date_columns].isnull().sum() 

Date            0
OrderDate       0
DeliveryDate    0
dtype: int64

In [15]:
#check delivery date is before order data
invalid_delivery_dates = df[df["DeliveryDate"] < df["OrderDate"]]

invalid_delivery_dates.shape

(0, 19)

In [16]:
#clean hidden spaces, inconsistent capitalization in text columns

text_columns = ["Region",
    "Product",
    "StoreLocation",
    "CustomerType",
    "Salesperson",
    "PaymentMethod",
    "Promotion",
    "OrderID",
    "CustomerName",
    "RegionManager"]

for col in text_columns:
        df[col] = df[col].astype(str).str.strip()

In [17]:
#keep promotion codes uppercase
df["Promotion"] = df["Promotion"].str.upper()
df["Promotion"] = df["Promotion"].replace("NO PROMOTION", "No Promotion")
print(df["Promotion"].head())

0    FREESHIP
1      SAVE10
2    WINTER15
3    FREESHIP
4      SAVE10
Name: Promotion, dtype: object


In [18]:
#see unique promotion values
print(df["Promotion"].unique())

['FREESHIP' 'SAVE10' 'WINTER15' 'No Promotion']


In [19]:
#correct  number formats
numeric_columns = ["Quantity", "UnitPrice", "Discount","TotalPrice", "ShippingCost"
]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [20]:
#to see whether any invalid values became NaN
df[numeric_columns].isnull().sum()

Quantity        0
UnitPrice       0
Discount        0
TotalPrice      0
ShippingCost    0
dtype: int64

In [21]:
#Round money columns to 2 decimals
df["UnitPrice"] = df["UnitPrice"].round(2)
df["TotalPrice"] = df["TotalPrice"].round(2)
df["ShippingCost"] = df["ShippingCost"].round(2)

In [22]:
#validate totalPrice

df["CalculatedTotal"] = df["Quantity"] * df["UnitPrice"] * (1 - df["Discount"])

df["CalculatedTotal"] = df["CalculatedTotal"].round(2)

price_errors = df[df["TotalPrice"] != df["CalculatedTotal"]]

price_errors.shape


(13, 20)

In [23]:
# View error rows
price_errors[[
    "OrderID",
    "Quantity",
    "UnitPrice",
    "Discount",
    "TotalPrice",
    "CalculatedTotal"
]]

,OrderID,Quantity,UnitPrice,Discount,TotalPrice,CalculatedTotal
281,REG100281,10,230.87,0.15,1962.40,1962.39
326,REG100326,5,311.22,0.15,1322.68,1322.69
548,REG100548,10,397.39,0.15,3377.82,3377.81
586,REG100586,5,440.93,0.10,1984.18,1984.19
697,REG100697,5,547.53,0.10,2463.89,2463.88
812,REG100812,9,160.15,0.10,1297.21,1297.22
818,REG100818,11,255.95,0.10,2533.91,2533.90
899,REG100899,15,324.58,0.15,4138.40,4138.39
960,REG100960,13,174.90,0.15,1932.64,1932.65
1134,REG101134,11,203.15,0.10,2011.18,2011.19


In [24]:
df = df.drop(columns=["CalculatedTotal"])

In [25]:
#validate  categories
for col in ["Region", "Product", "CustomerType", "PaymentMethod","Promotion","Returned"]:
    print(col)
    print(df[col].unique())   #check unique values
    print()

Region
['East' 'South' 'North' 'Central' 'West']

Product
['Laptop' 'Phone' 'Desk' 'Chair' 'Monitor' 'Tablet' 'Printer']

CustomerType
['Wholesale' 'Retail']

PaymentMethod
['Online' 'Gift Card' 'Credit Card' 'Debit Card' 'Cash']

Promotion
['FREESHIP' 'SAVE10' 'WINTER15' 'No Promotion']

Returned
[0 1]



In [26]:
#make Returned easier to understand
df["Returned"] = df["Returned"].replace({0:"No", 1:"Yes"})

In [27]:
#validation checklist
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDuplicate Order IDs:")
print(df["OrderID"].duplicated().sum())

print("\nInvalid delivery dates:")
print(df[df["DeliveryDate"] < df["OrderDate"]].shape[0])

print("\nDataset shape:")
print(df.shape)

Missing values:
Date             0
Region           0
Product          0
Quantity         0
UnitPrice        0
StoreLocation    0
CustomerType     0
Discount         0
Salesperson      0
TotalPrice       0
PaymentMethod    0
Promotion        0
Returned         0
OrderID          0
CustomerName     0
ShippingCost     0
OrderDate        0
DeliveryDate     0
RegionManager    0
dtype: int64

Duplicate rows:
0

Duplicate Order IDs:
0

Invalid delivery dates:
0

Dataset shape:
(1500, 19)


In [28]:
#save cleaned dataset
output_path= "output/Product-Sales-Region-Cleaned.xlsx"

df.to_excel("../output/Product-Sales-Region-Cleaned.xlsx",  index= False)

print("Cleaned file saved sucessfully!")

Cleaned file saved sucessfully!


In [29]:
df.to_csv("../output/Product-Sales-Region-Cleaned.csv", index=False)